# 🌡️ Análisis de Temperaturas — Tegucigalpa & San Pedro Sula

Este notebook lee los datos almacenados en PostgreSQL por el sistema de sensores,
los analiza y genera gráficos de comportamiento de la temperatura durante el día.


In [ ]:
# ── Dependencias ──────────────────────────────────────────────
import os
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from IPython.display import display

sns.set_theme(style='darkgrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print('Librerías cargadas ✔')

In [ ]:
# ── Conexión a la base de datos ───────────────────────────────
conn = psycopg2.connect(
    host=os.environ.get('DB_HOST', 'db'),
    port=os.environ.get('DB_PORT', 5432),
    dbname=os.environ.get('DB_NAME', 'clima_db'),
    user=os.environ.get('DB_USER', 'clima_user'),
    password=os.environ.get('DB_PASS', 'clima_pass'),
)
print('Conexión exitosa ✔')

In [ ]:
# ── Cargar datos ──────────────────────────────────────────────
df = pd.read_sql("""
    SELECT id, ciudad, temperatura, unidad, latitud, longitud,
           instancia_api, fecha_hora
    FROM temperatura_lecturas
    ORDER BY fecha_hora ASC
""", conn)

df['fecha_hora'] = pd.to_datetime(df['fecha_hora'])
df['hora']       = df['fecha_hora'].dt.floor('min')
df['fecha']      = df['fecha_hora'].dt.date

print(f'Total de registros: {len(df)}')
display(df.tail(10))

In [ ]:
# ── Estadísticas por ciudad ───────────────────────────────────
resumen = df.groupby('ciudad')['temperatura'].agg(
    Minima='min', Maxima='max', Promedio='mean', Lecturas='count'
).round(2)

print('\n📊 Resumen estadístico por ciudad:')
display(resumen)

In [ ]:
# ── Gráfico 1: Temperatura durante el día (por ciudad) ────────
ciudades  = df['ciudad'].unique()
colores   = {'Tegucigalpa': '#e74c3c', 'San Pedro Sula': '#2980b9'}

fig, ax = plt.subplots(figsize=(12, 5))

for ciudad in ciudades:
    sub = df[df['ciudad'] == ciudad]
    color = colores.get(ciudad, 'green')
    ax.plot(sub['fecha_hora'], sub['temperatura'],
            marker='o', markersize=4, linewidth=1.8,
            label=ciudad, color=color)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
fig.autofmt_xdate()

ax.set_title('Temperatura durante el día — Tegucigalpa vs San Pedro Sula', fontsize=14, pad=12)
ax.set_xlabel('Hora del día')
ax.set_ylabel('Temperatura (°C)')
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('grafico_temperatura_dia.png', bbox_inches='tight')
plt.show()
print('Gráfico guardado: grafico_temperatura_dia.png')

In [ ]:
# ── Gráfico 2: Comparativa min/max/promedio por ciudad ────────
fig, ax = plt.subplots(figsize=(8, 5))

x      = range(len(resumen))
width  = 0.25
cities = resumen.index.tolist()

bars1 = ax.bar([i - width for i in x], resumen['Minima'],  width, label='Mínima',  color='#3498db', alpha=0.85)
bars2 = ax.bar([i         for i in x], resumen['Promedio'],width, label='Promedio', color='#2ecc71', alpha=0.85)
bars3 = ax.bar([i + width for i in x], resumen['Maxima'],  width, label='Máxima',  color='#e74c3c', alpha=0.85)

ax.set_xticks(list(x))
ax.set_xticklabels(cities)
ax.set_ylabel('Temperatura (°C)')
ax.set_title('Resumen de Temperaturas por Ciudad', fontsize=14, pad=12)
ax.legend()

# Etiquetas sobre las barras
for bar in [*bars1, *bars2, *bars3]:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('grafico_resumen_ciudades.png', bbox_inches='tight')
plt.show()
print('Gráfico guardado: grafico_resumen_ciudades.png')

In [ ]:
# ── Gráfico 3 (EXTRA): Peticiones por instancia de API ───────
instancias = df['instancia_api'].value_counts().reset_index()
instancias.columns = ['instancia', 'peticiones']

fig, ax = plt.subplots(figsize=(6, 4))
colors  = ['#9b59b6', '#e67e22', '#1abc9c']

ax.bar(instancias['instancia'], instancias['peticiones'],
       color=colors[:len(instancias)], alpha=0.9, edgecolor='white')

for i, row in instancias.iterrows():
    ax.text(i, row['peticiones'] + 0.3, str(row['peticiones']),
            ha='center', fontweight='bold')

ax.set_title('Distribución de Peticiones por Instancia (Load Balancer)', fontsize=12)
ax.set_xlabel('Instancia de API')
ax.set_ylabel('Número de peticiones atendidas')
plt.tight_layout()
plt.savefig('grafico_balanceo_carga.png', bbox_inches='tight')
plt.show()
print('Gráfico guardado: grafico_balanceo_carga.png')
print('\nDistribución:', instancias.to_dict('records'))

In [ ]:
# ── Cierre de conexión ────────────────────────────────────────
conn.close()
print('Análisis completo ✔')